# పాఠం 11 - ఏజెంట్-టు-ఏజెంట్ (A2A) ప్రోటోకాల్


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A ప్రోటోకాల్ అంటే ఏమిటి?

The **ఏజెంట్-టు-ఏజెంట్ (A2A) ప్రోటోకాల్** ఒక ఓపెన్ స్టాండర్డ్‌, ఇది AI ఏజెంట్స్‌కు పరస్పరం కమ్యూనికేట్ చేయడం, ఒకరిని కనుగొనడం, మరియు కలిసి సహకరించడం సాధ్యమవిస్తుంది — అవి వేర్వేరు ఫ్రేమ్‌వర్క్‌లపై నిర్మించబడ్డా లేదా వేర్వేరు సేవల ద్వారా హోస్ట్ చేయబడుతున్నా కూడా.

Key concepts:

- **అన్వేషణ** – ఏజెంట్స్ వారి సామర్థ్యాలను వివరించే ఒక *ఏజెంట్ కార్డ్* ప్రచురిస్తాయి, తద్వారా ఇతర ఏజెంట్స్ (లేదా ఆర్కెస్ట్రేటర్లు) ఒక పనికి సరైన ప్రత్యేక నిపుణుడిని కనుగొనడం సులభం అవుతుంది.
- **సందేశ మార్పిడీ** – ఏజెంట్స్ ఒక సాధారణ ప్రోటోకాల్ ద్వారా నిర్మిత సందేశాలను మార్పిడి చేస్తాయి, అందువల్ల ఒక ఏజెంట్ నుంచి వచ్చిన అభ్యర్థనను మరో ఏజెంట్ వారి అంతర్గత అమలాపై ఆధారపడకుండా అర్థం చేసుకుని పూర్తిచేయగలదు.
- **టాస్క్ జీవ చక్రం** – A2A *సమర్పించబడింది*, *పనిలో ఉంది*, *పూర్తయింది*, మరియు *విఫలమైంది* వంటి స్థితులను నిర్వచిస్తుంది, తద్వారా ఆర్కెస్ట్రేటర్‌కు అప్పగించిన పని ఎలా పురోగమిస్తున్నదో పూర్తిగా కనిపిస్తుంది.

ఈ పాఠంలో మేము A2A-శైలి సహకారాన్ని అనుకరించి మూడు ప్రత్యేక ప్రయాణ ఏజెంట్లను ఒక వర్క్‌ఫ్లోలో అనుసంధానించడం ద్వారా, ప్రతి ఏజెంట్ తన నైపుణ్యాన్ని అందించి ఫలితాలను తదుపరి ఏజెంట్కి పంపే విధానాన్ని చూపిస్తాం.


## ప్రత్యేకమైన ప్రయాణ ఏజెంట్లను సృష్టించడం


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## వర్క్‌ఫ్లో ద్వారా బహుళ ఏజెంట్ సహకారం

మేము ఆ మూడు ఏజెంట్లను A2A సందేశ పంపకాన్ని ప్రతిబింబించే అనుక్రమ వర్క్‌ఫ్లోగా అనుసంధానిస్తాము:

1. **CurrencyExchangeAgent** వినియోగదారు అభ్యర్థనను స్వీకరించి కరెన్సీ మార్గదర్శకత్వాన్ని రూపొందిస్తుంది.
2. **ActivityPlannerAgent** మెరుగుపరచబడిన సందర్భాన్ని స్వీకరించి కార్యకలాప సిఫార్సులను జోడిస్తుంది.
3. **TravelManagerAgent** రెండు ఇన్‌పుట్‌లను సంయోజించి తుది ప్రయాణ సారాంశంగా రూపొందిస్తుంది.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ఉత్పత్తిలో A2A ను అర్థం చేసుకోవడం

ఉత్పత్తి పరిసరాల్లో A2A ప్రోటోకాల్ శక్తివంతమైన క్రాస్-సర్వీస్ సన్నివేశాలను అనుమతిస్తుంది:

| Capability | Description |
|---|---|
| **ఫ్రేమ్‌వర్క్‌ల మధ్య పరస్పర కార్యాచరణ** | ఒక ఫ్రేమ్‌వర్క్‌తో నిర్మించబడిన ఏజెంట్ ఇతర ఏదైనా A2A-సమర్ధ ఫ్రేమ్‌వర్క్‌లో నిర్మించబడిన ఏజెంట్‌కు పనులను అప్పగించగలడు, తద్వారా నిజమైన సంస్థల మధ్య పరస్పర కార్యసాధ్యం సాధ్యమవుతుంది. |
| **సేవా సరిహద్దులు** | ఏజెంట్లు వేర్వేరు మైక్రోసర్వీసులు, క్లౌడ్ ప్రాంతాలు, లేదా వేర్వేరు సంస్థలలో ఉండగలవు కానీ అవి సాఫీగా కలిసి పనిచేస్తాయి. |
| **డైనమిక్ కనుగొనడం** | ఒక ఆర్కెస్ట్రేటర్ ర‌న్‌టైమ్‌లో Agent Card రిజిస్ట్రీని ప్రశ్నించి నిర్దిష్ట ఉప-పనికి అత్యుత్తమ అనుకూల నిపుణుణ్ని కనుగొనవచ్చు. |
| **స్ట్రీమింగ్ & పుష్ నోటిఫికేషన్లు** | A2A రియల్-టైమ్ పురోగతి నవీకరణల కోసం Server-Sent Events (SSE)నీ, అలాగే దీర్ఘకాలిక పనుల కోసం పుష్ నోటిఫికేషన్లను మద్దతు ఇస్తుంది. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## సారాంశం

ఈ పాఠంలో మీరు నేర్చుకున్నారు:

1. **A2A ప్రోటోకాల్ అంటే ఏమిటి** — ఏజెంట్-తొ-ఏజెంట్ అన్వేషణ, మెసేజింగ్, మరియు పని నిర్వహణ కోసం ఓపెన్ స్టాండర్డ్.
2. **ప్రత్యేక ఏజెంట్లను ఎలా సృష్టించాలి** — ఒక Currency Exchange ఏజెంట్, ఒక Activity Planner ఏజెంట్, మరియు ఒక Travel Manager ఆర్కెస్ట్రేటర్.
3. **ఏజెంట్లను ఒక వర్క్‌ఫ్లోలో ఎలా వైర్ చేయాలి** — ఏజెంట్ల మధ్య వరుసగా సందేశాల పంపకాన్ని మోడల్ చేయడానికి `WorkflowBuilder` ను ఉపయోగించడం.
4. **A2A ఉత్పత్తి పరిసరంలో ఎలా పనిచేస్తుంది** — డైనమిక్ అన్వేషణ మరియు స్ట్రీమింగ్ అప్డేట్లతో విభిన్న ఫ్రేమ్‌వర్క్‌లు, విభిన్న సర్వీస్‌లు మధ్య సహకారాన్ని సులభతరం చేయడం.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
నిరాకరణ:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి శ్రమిస్తూన్నప్పటికీ, స్వయంచాలక అనువాదాల్లో తప్పులు లేదా పొరపాట్లు ఉండవచ్చని దయచేసి గమనించండి. మూల భాషలో ఉన్న అసలైన పత్రాన్ని అధికారిక మూలంగా తీసుకోవాలి. కీలకత ఉన్న సమాచారం కోసం, వృత్తిపరమైన మానవ అనువాదం చేయించుకోవాలని సూచిస్తాము. ఈ అనువాదాన్ని ఉపయోగించాక కలిగే ఏదైనా అపార్థాలు లేదా తప్పుగా అర్థం చేసుకోవడాల గురించి మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
